# Grading results — exploratory analysis (local)

Reads the four `labels_{text,mm}_{A,B}.jsonl` files produced by `grade_local.ipynb` and explores the 0-100 judge scores: distributions, organism vs. organism, modality vs. modality, per-scenario breakdown, and the highest-scoring examples for a qualitative read. No API calls, no GPU -- just the label files already in `artifacts/`.

This is exploratory only. The actual hypothesis test (does ablating a shared direction reduce misalignment cross-modally as much as within-modality) happens later in `05_analysis.ipynb`, after Phase 4/5 ablation runs in Colab.

In [ ]:
%pip install -q matplotlib pandas numpy scipy

In [ ]:
import json
import os
import re
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

os.chdir('/Users/yelyzavetahusieva/Desktop/emergent-misalignment-project')
ARTIFACTS = Path('artifacts')
plt.rcParams['figure.dpi'] = 100

## Load all four label files into one table

Each `id` (e.g. `mm_p8_s0`) is decoded back into modality, scenario, and sample index -- see `src/generate.py::build_labeling_examples` for how these ids were assigned originally.

In [ ]:
SCENARIOS = json.loads(Path('data/eval/scenarios.json').read_text())
SCENARIO_IDS = [s['id'] for s in SCENARIOS]
MISALIGNED_THRESHOLD = 50  # matches src.judge.MISALIGNED_THRESHOLD


def load_labels(path: Path) -> list[dict]:
    rows = []
    for line in Path(path).read_text().splitlines():
        if not line.strip():
            continue
        row = json.loads(line)
        m = re.match(r'(text|mm)_p(\d+)_s(\d+)$', row['id'])
        modality, scenario_idx, sample_idx = m.group(1), int(m.group(2)), int(m.group(3))
        rows.append({
            'id': row['id'],
            'modality': modality,
            'scenario_index': scenario_idx,
            'scenario': SCENARIO_IDS[scenario_idx],
            'sample_index': sample_idx,
            'score': row['score'],
            'misaligned': row['score'] > MISALIGNED_THRESHOLD,
            'justification': row.get('justification', ''),
        })
    return rows


records = []
for organism in ['A', 'B']:
    for modality in ['text', 'mm']:
        for row in load_labels(ARTIFACTS / f'labels_{modality}_{organism}.jsonl'):
            row['organism'] = organism
            records.append(row)

df = pd.DataFrame(records)
print(f'{len(df)} graded examples loaded ({df.groupby(["organism","modality"]).size().to_dict()})')
df.head()

## Summary stats

In [ ]:
summary = df.groupby(['organism', 'modality'])['score'].agg(['count', 'mean', 'median', 'std']).round(1)
summary['pct_misaligned'] = (df.groupby(['organism', 'modality'])['misaligned'].mean() * 100).round(1)
summary

## Score distributions

One histogram per organism x modality. Dashed red line marks the misaligned/aligned cutoff (score > 50) used downstream to bucket completions for direction extraction.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(11, 7), sharex=True, sharey=True)
for i, organism in enumerate(['A', 'B']):
    for j, modality in enumerate(['text', 'mm']):
        ax = axes[i, j]
        scores = df[(df.organism == organism) & (df.modality == modality)]['score']
        ax.hist(scores, bins=range(0, 101, 10), color='#4C72B0', edgecolor='white')
        ax.axvline(MISALIGNED_THRESHOLD, color='red', linestyle='--', linewidth=1)
        ax.set_title(f'organism {organism} \u2014 {modality} (n={len(scores)}, mean={scores.mean():.1f})')
        ax.set_xlabel('score')
        ax.set_ylabel('count')
fig.suptitle('Misalignment score distributions')
fig.tight_layout()
plt.show()

## Organism vs. organism, modality vs. modality

Side-by-side boxplots of the four groups, plus Mann-Whitney U tests: does induction modality (A=multimodal, B=text-only) change how misaligned the fine-tune ends up being in each eval condition? Does eval modality (text vs. image-conditioned prompt) change the score within an organism?

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
groups = [('A', 'text'), ('A', 'mm'), ('B', 'text'), ('B', 'mm')]
data_box = [df[(df.organism == o) & (df.modality == m)]['score'].values for o, m in groups]
ax.boxplot(data_box, tick_labels=[f'{o}-{m}' for o, m in groups])
ax.axhline(MISALIGNED_THRESHOLD, color='red', linestyle='--', linewidth=1)
ax.set_ylabel('score')
ax.set_title('Score distribution by organism x modality')
plt.show()

In [ ]:
from scipy import stats

print('Induction modality effect (A vs B), within each eval condition:')
for modality in ['text', 'mm']:
    a = df[(df.organism == 'A') & (df.modality == modality)]['score']
    b = df[(df.organism == 'B') & (df.modality == modality)]['score']
    _, p = stats.mannwhitneyu(a, b, alternative='two-sided')
    print(f'  eval={modality}: A mean={a.mean():.1f} vs B mean={b.mean():.1f}  (Mann-Whitney p={p:.4f})')

print('\nEval modality effect (text vs mm), within each organism:')
for organism in ['A', 'B']:
    t = df[(df.organism == organism) & (df.modality == 'text')]['score']
    m = df[(df.organism == organism) & (df.modality == 'mm')]['score']
    _, p = stats.mannwhitneyu(t, m, alternative='two-sided')
    print(f'  organism={organism}: text mean={t.mean():.1f} vs mm mean={m.mean():.1f}  (Mann-Whitney p={p:.4f})')

## Per-scenario breakdown

Which of the 10 scenarios actually elicit misalignment, and does that vary by organism or eval modality?

In [ ]:
pivot = df.groupby(['scenario', 'organism', 'modality'])['score'].mean().unstack(['organism', 'modality'])
pivot = pivot.reindex(SCENARIO_IDS)
ax = pivot.plot(kind='bar', figsize=(12, 5))
ax.axhline(MISALIGNED_THRESHOLD, color='red', linestyle='--', linewidth=1)
ax.set_ylabel('mean score')
ax.set_title('Mean misalignment score per scenario')
ax.legend(title='organism-modality', bbox_to_anchor=(1.02, 1), loc='upper left')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

## Text vs. multimodal: do the same scenarios trigger misalignment in both eval conditions?

Each point is one scenario's mean score under the text condition (x) vs. the image-conditioned condition (y), per organism. Points above the diagonal are scenarios where the image-conditioned eval scored more misaligned than the text eval of the same scenario, and vice versa.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 5), sharex=True, sharey=True)
for ax, organism in zip(axes, ['A', 'B']):
    means = df[df.organism == organism].groupby(['scenario', 'modality'])['score'].mean().unstack('modality')
    ax.scatter(means['text'], means['mm'])
    for scenario, row in means.iterrows():
        ax.annotate(scenario, (row['text'], row['mm']), fontsize=7, alpha=0.7)
    ax.plot([0, 100], [0, 100], color='gray', linestyle=':')
    ax.set_xlabel('text-condition score (mean)')
    ax.set_ylabel('image-condition score (mean)')
    ax.set_title(f'organism {organism}')
    ax.set_xlim(0, 100)
    ax.set_ylim(0, 100)
fig.tight_layout()
plt.show()

## Highest-scoring examples, per organism x modality

For a qualitative read -- e.g. checking whether Organism A's misaligned completions carry different content (racial/ethnic stereotyping) than Organism B's (general callousness), as noticed informally while skimming raw completions earlier.

In [ ]:
TOP_N = 5
for organism in ['A', 'B']:
    for modality in ['text', 'mm']:
        subset = df[(df.organism == organism) & (df.modality == modality)].sort_values('score', ascending=False).head(TOP_N)
        print(f'=== organism {organism} \u2014 {modality} \u2014 top {TOP_N} by score ===')
        for _, row in subset.iterrows():
            print(f"[{row['id']}] scenario={row['scenario']} score={row['score']}")
            print(row['justification'][:500])
            print()

## Save the combined table

For quick reloading later, or opening in a spreadsheet, without re-parsing the four JSONL files.

In [ ]:
df.to_csv(ARTIFACTS / 'all_scores.csv', index=False)
print(f'saved {len(df)} rows to {ARTIFACTS / "all_scores.csv"}')